In [1]:
import h5py
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.ndimage import label
from skimage import morphology
from skimage import measure
from scipy import ndimage as ndi
from skimage.segmentation import find_boundaries
from scipy.stats import gaussian_kde
from matplotlib.patches import FancyArrowPatch
from scipy.optimize import curve_fit
from scipy.ndimage import map_coordinates
import seaborn as sns
from scipy.ndimage import maximum_filter, label
from scipy.stats import skewnorm
from scipy.stats import lognorm
import fifpy
from scipy.ndimage import gaussian_filter
from scipy.ndimage import label, gaussian_filter
from scipy.stats import lognorm, gaussian_kde

# shared code and paths (see README.md and shared/)
import sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "paths.py").exists())
sys.path.insert(0, str(REPO))
from shared.detection import detect_current_sheets
from shared.fields import partial_x_fft, partial_y_fft
from shared.filtering import iterative_filtering1D, iterative_filtering3D
from shared.paths import DATA_ROOT, NAS_ROOT, results_dir, cache_dir as shared_cache_dir
RESULTS = results_dir("cs_identification")
CACHE = shared_cache_dir("cs_identification")

In [2]:
#Defining all the required functions

#Fourier Transform Functions


#Defining constants
dx = 0.0625
dy = 0.0625





In [ ]:
# -----------------------------
# USER SETTINGS
# -----------------------------
beta = 0.25                  # pick one: 0.0625, 1, or 16
times = [200]             # add more snapshots here, e.g. [100, 150, 200]
cache_dir = str(CACHE / "filtered_cache_Jz")
os.makedirs(cache_dir, exist_ok=True)

# storage (keyed by time)
J2_all = {}
mask_all = {}

# ----- FILTER PARAMETERS (depend on beta) -----
if beta == 0.0625:
    n_iter, Xi = 3, 2.0
elif beta == 0.25:
    n_iter, Xi = 3, 2.3
elif beta == 1:
    n_iter, Xi = 3, 2.5
elif beta == 4:
    n_iter, Xi = 3, 2.7
else:
    n_iter, Xi = 3, 3.0

base_path = f'{DATA_ROOT}/beta_p_{beta}/beta_e_1/'

# -----------------------------
# MAIN LOOP (over times only)
# -----------------------------
for time in times:
    print(f"\n--- Processing beta = {beta}, t = {time} ---")

    File_Bx = base_path + f'Bx_ApJ_t{time}.h5'
    File_By = base_path + f'By_ApJ_t{time}.h5'
    File_Bz = base_path + f'Bz_ApJ_t{time}.h5'

    with h5py.File(File_Bx, 'r') as fBx, \
         h5py.File(File_By, 'r') as fBy, \
         h5py.File(File_Bz, 'r') as fBz:
        data_Bx = fBx['DS1'][:].T
        data_By = fBy['DS1'][:].T
        data_Bz = fBz['DS1'][:].T

    cache_file = f"{cache_dir}/filtered_beta_{beta}_t{time}_iter{n_iter}_Xi{Xi}.npz"

    if os.path.exists(cache_file):
        print("Loading cached filtered fields...")
        data = np.load(cache_file)
        data_Bx_f = data['Bx']
        data_By_f = data['By']
        data_Bz_f = data['Bz']
    else:
        print("Computing filtering (slow)...")
        data_Bx_f, data_By_f, data_Bz_f = iterative_filtering3D(
            data_Bx, data_By, data_Bz, n_iter, Xi=Xi
        )
        np.savez_compressed(
            cache_file, Bx=data_Bx_f, By=data_By_f, Bz=data_Bz_f
        )

    # ----- CURRENT DENSITY -----
    Jx = partial_y_fft(data_Bz_f, dx)
    Jy = -partial_x_fft(data_Bz_f, dy)
    Jz = partial_x_fft(data_By_f, dy) - partial_y_fft(data_Bx_f, dx)
    # J2 = Jx**2 + Jy**2 + Jz**2
    J2 = np.abs(Jz)

    plt.hist(J2.ravel(), bins=200, log=True,
             histtype='step', linewidth=2,
             label=fr'$\beta = {beta}$, $t = {time}$')

    # ----- CURRENT SHEET DETECTION -----
    sheets = detect_current_sheets(J2)
    mask = np.zeros_like(J2)
    for sheet in sheets:
        for (i, j) in sheet:
            mask[i, j] = 1

    # ----- MORPHOLOGICAL CLEANING -----
    struct = morphology.disk(1)
    mask = morphology.binary_opening(mask, struct)
    mask = morphology.binary_closing(mask, struct)
    mask = morphology.remove_small_objects(
        mask.astype(bool), min_size=500
    ).astype(int)

    J2_all[time] = J2
    mask_all[time] = mask

plt.legend()
plt.xlabel(r'$J^2$')
plt.ylabel('Count')
plt.title(fr'Distribution of $J^2$ for $\beta = {beta}$')
plt.show()

# -----------------------------
# COLOR SCALE
# -----------------------------
all_values = np.concatenate([v.ravel() for v in J2_all.values()])
vmax_global = np.percentile(all_values, 99)

# -----------------------------
# PLOT (one row of panels, one per time)
# -----------------------------
fig, axes = plt.subplots(
    1, len(times),
    figsize=(6 * len(times), 6),
    squeeze=False
)

for c, time in enumerate(times):
    ax = axes[0][c]
    im = ax.imshow(
        J2_all[time].T, cmap='inferno', origin='lower', vmax=vmax_global
    )
    ax.contour(
        mask_all[time].T, levels=[0.5], colors="cyan", linewidths=0.4
    )
    ax.set_title(fr'$\beta = {beta}$, $t = {time}$')
    ax.set_xlabel(r'$x/d_p$')
    ax.set_ylabel(r'$y/d_p$')

plt.tight_layout()
plt.savefig(RESULTS / f"J2_comparison_beta_{beta}.png", dpi=300)
plt.show()


--- Processing beta = 0.25, t = 200 ---
Loading cached filtered fields...


In [18]:
cleaned_mask = mask_all[time]
J2 = J2_all[time]
labeled_mask, num_features = label(cleaned_mask)
print(f"[t={time}] Regions found:", num_features)

[t=200] Regions found: 394


In [3]:
# ============================================================
#  Animation of current sheets over time (t = 0 .. 300)
#  ------------------------------------------------------------
#  1. precompute every frame in parallel: load B, filter, |Jz|, detect sheets
#  2. build an animation (|Jz| map + sheet outline) from the stored frames
#  Saves an mp4 (or a gif if ffmpeg is not available).
#
#  Needs from your session: iterative_filtering3D, partial_x_fft, partial_y_fft,
#                            detect_current_sheets
# ============================================================

import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import h5py
from skimage import morphology
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# -----------------------------
# SETTINGS
# -----------------------------
beta       = 0.0625
frame_step = 1                     # use every Nth snapshot (raise to 2/5 to speed up)
times      = list(range(1, 301, frame_step))
sim_tag    = "ApJ"
base_path  = f"{DATA_ROOT}/beta_p_{beta}/beta_e_1/"
cache_dir  = str(CACHE / "filtered_cache_Jz")
out_movie  = str(RESULTS / f"cs_movie_beta{beta}.mp4")
fps        = 12
n_workers  = min(len(times), os.cpu_count() or 1)
dx = dy    = 0.0625

os.makedirs(cache_dir, exist_ok=True)

# filter parameters for this beta
if beta == 0.0625:
    n_iter, Xi = 3, 2.0
elif beta == 0.25:
    n_iter, Xi = 3, 2.3
elif beta == 1:
    n_iter, Xi = 3, 2.5
elif beta == 4:
    n_iter, Xi = 3, 2.7
else:
    n_iter, Xi = 3, 3.0


# -----------------------------
# One frame: load -> filter (cached) -> |Jz| -> detect sheets
# returns (time, J2, mask)  or  None if the snapshot is missing
# -----------------------------
def compute_frame(time):
    fBx = base_path + f"Bx_{sim_tag}_t{time}.h5"
    fBy = base_path + f"By_{sim_tag}_t{time}.h5"
    fBz = base_path + f"Bz_{sim_tag}_t{time}.h5"
    if not (os.path.exists(fBx) and os.path.exists(fBy) and os.path.exists(fBz)):
        return None

    with h5py.File(fBx, "r") as hBx, h5py.File(fBy, "r") as hBy, h5py.File(fBz, "r") as hBz:
        data_Bx = hBx["DS1"][:].T
        data_By = hBy["DS1"][:].T
        data_Bz = hBz["DS1"][:].T

    cache_file = f"{cache_dir}/filtered_beta_{beta}_t{time}_iter{n_iter}_Xi{Xi}.npz"
    if os.path.exists(cache_file):
        d = np.load(cache_file)
        data_Bx_f, data_By_f, data_Bz_f = d["Bx"], d["By"], d["Bz"]
    else:
        data_Bx_f, data_By_f, data_Bz_f = iterative_filtering3D(
            data_Bx, data_By, data_Bz, n_iter, Xi=Xi)
        np.savez_compressed(cache_file, Bx=data_Bx_f, By=data_By_f, Bz=data_Bz_f)

    Jz = partial_x_fft(data_By_f, dy) - partial_y_fft(data_Bx_f, dx)
    J2 = np.abs(Jz)

    sheets = detect_current_sheets(J2)
    mask = np.zeros_like(J2)
    for sheet in sheets:
        for (i, j) in sheet:
            mask[i, j] = 1
    struct = morphology.disk(1)
    mask = morphology.binary_opening(mask, struct)
    mask = morphology.binary_closing(mask, struct)
    mask = morphology.remove_small_objects(mask.astype(bool), min_size=500).astype(int)

    # store compactly: float32 field + boolean mask
    return time, J2.astype(np.float32), mask.astype(bool)


# -----------------------------
# STEP 1. Precompute all frames (parallel)
# -----------------------------
frames = []
if n_workers == 1:
    for t in times:
        r = compute_frame(t)
        if r is not None:
            frames.append(r)
            print(f"frame t={t} done", flush=True)
else:
    ctx = mp.get_context("fork")
    print(f"Computing {len(times)} frames across {n_workers} workers ...", flush=True)
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=ctx) as ex:
        futs = {ex.submit(compute_frame, t): t for t in times}
        done = 0
        for fut in as_completed(futs):
            r = fut.result()
            done += 1
            if r is not None:
                frames.append(r)
            print(f"[{done}/{len(times)}] frame t={futs[fut]} done", flush=True)

frames.sort(key=lambda f: f[0])          # as_completed returns out of order -> sort by time
print(f"{len(frames)} frames ready", flush=True)
if len(frames) == 0:
    raise SystemExit("No frames were produced (check paths / times).")

# fixed colour scale across all frames, so the movie doesn't flicker
vmax = max(float(np.percentile(J2, 99)) for _, J2, _ in frames)


# -----------------------------
# STEP 2. Build and save the animation
# -----------------------------
fig, ax = plt.subplots(figsize=(7, 6))
im0 = ax.imshow(frames[0][1].T, cmap="inferno", origin="lower", vmin=0, vmax=vmax)
fig.colorbar(im0, ax=ax, label=r"$|J_z|$")

def update(i):
    t, J2, mask = frames[i]
    ax.clear()
    ax.imshow(J2.T, cmap="inferno", origin="lower", vmin=0, vmax=vmax)
    ax.contour(mask.T, levels=[0.5], colors="cyan", linewidths=0.5)
    ax.set_title(fr"$\beta = {beta}$,   $t = {t}$")
    ax.set_xlabel(r"$x/d_p$")
    ax.set_ylabel(r"$y/d_p$")

anim = FuncAnimation(fig, update, frames=len(frames), interval=1000 / fps)

# save as mp4 if ffmpeg is present, otherwise fall back to gif
try:
    anim.save(out_movie, writer="ffmpeg", fps=fps, dpi=150)
    print(f"saved {out_movie}", flush=True)
except Exception as e:
    out_movie = out_movie.replace(".mp4", ".gif")
    anim.save(out_movie, writer="pillow", fps=fps, dpi=100)
    print(f"ffmpeg unavailable ({type(e).__name__}); saved {out_movie} instead", flush=True)

plt.close(fig)

Computing 300 frames across 128 workers ...
IMF 1IMF
 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF IMF1
 1
IMF 1
IMF IMF1
 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF IMF1 
1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMFIMF  11

IMF 1
IMFIMF 1
 1
IMF IMF1 
IMF 1
IMF1 1

IMF IMF 1
1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1
IMF 1IMF
 1
IMF 1
IMF 1
IMF 1
IMF 1IMF 
1
IMF 1IMF
 1
IMF IMF1
 IMF1IMF  11


IMF 1
IMF 1
IMF 1
IMFIMFIMF  1 IMFIMF
 1 1

11

IMFIMF  11

IMFIMF  1
1
IMF 1IMFIMF
 IMF1IMF 
 IMF 11

IMFIMF  1 1IMFIMF
IMF  1

1IMF
1 
1 
11

IMFIMF 1
 1
IMFIMFIMF 1
  11IMF

IMF 1IMF  1
1

IMF 1
IMF IMFIMF 1 1
1

IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2IMF
 2
IMF 2
IMF 2
IMF 2
IMFIMF 2
 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2IMF
 2
IMF 2
IMF 2
IMF 2
IMF 2
IMF 2IMF
 2
IMFIMF 2
IMF  2
2
IMF 2
IMF 2
IMF 2
IMF 2
IMF IMF2 
2
IMF 2IMF
 2
IMF 2
IMFIMF  2
2IMF 
2
IMF 2
IM